# CBIS-DDSM: Mammography Image Quality Assessment (IQA) & Adaptive Routing
### Modality: Screening Mammography (X-ray Projection Radiography: CC & MLO Views)
### Dataset: Curated Breast Imaging Subset of DDSM (CBIS-DDSM)

---

## 📌 Clinical & Radiographic Objectives
1. **Dataset Characterization:** Map mammography cohort cases, screening views (`CC` - Craniocaudal, `MLO` - Mediolateral Oblique), abnormality categories (`Masses` vs `Microcalcifications`), and pathology ground truth (`BENIGN`, `MALIGNANT`).
2. **Identification of Inherent Challenges:** Quantify digitized film grain, scanner dust/scratches, dense breast parenchymal obscuration, and high anatomical dynamic range.
3. **Multi-Metric IQA Pipeline:**
   - **Sharpness:** Laplacian Variance ($\sigma^2_{\Delta}$), Tenengrad Gradient
   - **Contrast & Dynamic Spread:** RMS Contrast ($C_{\text{RMS}}$), Michelson Dynamic Range
   - **Tissue Content:** Shannon Information Entropy ($H$)
   - **Noise Estimation:** Blind SNR via Median Absolute Deviation (MAD) of high frequencies
   - **Dense Tissue Ratio:** Proportion of fibroglandular parenchyma (BI-RADS density proxy)
4. **Statistical Analysis:** Generate descriptive summaries across abnormality types and views, correlation heatmaps, and side-by-side visual montages.
5. **Learned Decision Tree:** Train an explainable adaptive routing model that evaluates image fidelity + CNN prediction confidence before triggering automated diagnostic sign-off or radiologist review.

---
## 🚀 Running on Kaggle
- Click **"Add Input"** on the right sidebar and search for `CBIS-DDSM` or `awsaf49/cbis-ddsm-breast-cancer-image-dataset`.
- The notebook auto-detects `/kaggle/input/` paths and writes outputs to `/kaggle/working/`.


In [ ]:
# Cell 1: Environment Setup & Library Imports
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from scipy.stats import entropy
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
print("Environment and libraries loaded successfully.")


In [ ]:
# Cell 2: Dataset Discovery & Path Configuration
KAGGLE_CANDIDATE_PATHS = [
    "/kaggle/input/cbis-ddsm-breast-cancer-image-dataset",
    "/kaggle/input/cbis-ddsm",
    "/kaggle/input/ddsm-mammography",
    "./sample_data",
    "../sample_data",
]

def locate_cbis_dataset():
    for p in KAGGLE_CANDIDATE_PATHS:
        if os.path.exists(p) and len(os.listdir(p)) > 0:
            print(f"[SUCCESS] CBIS-DDSM located at: {p}")
            return p
    print("[NOTICE] No external Kaggle dataset mounted. Initializing sample mammography cohort...")
    return None

DATASET_ROOT = locate_cbis_dataset()
OUTPUT_DIR = "/kaggle/working/output" if os.path.exists("/kaggle") else "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Cell 3: Synthetic Mammography Generator (Zero-Error Fallback)
def generate_synthetic_mammograms(target_dir, n_samples=60):
    os.makedirs(target_dir, exist_ok=True)
    views = ["CC", "MLO"]
    pathologies = ["BENIGN", "MALIGNANT"]
    abnormalities = ["Mass", "Calcification"]
    np.random.seed(42)

    for i in range(n_samples):
        view = views[i % 2]
        pathology = pathologies[(i // 2) % 2]
        abnormality = abnormalities[(i // 4) % 2]
        sub = os.path.join(target_dir, f"{pathology}_{abnormality}")
        os.makedirs(sub, exist_ok=True)

        h, w = 384, 256
        img = np.zeros((h, w), dtype=np.uint8)

        # Breast outline arc
        center = (0, int(h * 0.5))
        axes = (int(w * 0.90), int(h * 0.45))
        cv2.ellipse(img, center, axes, 0, -90, 90, 85, -1)

        # Pectoral muscle on MLO view
        if view == "MLO":
            pts = np.array([[0, 0], [int(w * 0.45), 0], [0, int(h * 0.40)]], np.int32)
            cv2.fillPoly(img, [pts], 160)

        # Dense fibroglandular parenchyma
        dense = np.zeros((h, w), dtype=np.uint8)
        cv2.ellipse(dense, center, (int(w * 0.65), int(h * 0.35)), 0, -90, 90, 50, -1)
        dense = cv2.GaussianBlur(dense, (35, 35), 0)
        img = cv2.add(img, dense)

        # Abnormality
        cx = np.random.randint(int(w * 0.25), int(w * 0.70))
        cy = np.random.randint(int(h * 0.30), int(h * 0.70))

        if abnormality == "Mass":
            r = np.random.randint(12, 22)
            if pathology == "MALIGNANT":
                cv2.circle(img, (cx, cy), r, 220, -1)
                for _ in range(8):
                    ang = np.random.uniform(0, 2 * np.pi)
                    cv2.line(img, (cx, cy), (int(cx + (r + 10) * np.cos(ang)), int(cy + (r + 10) * np.sin(ang))), 210, 2)
            else:
                cv2.ellipse(img, (cx, cy), (r, int(r * 0.7)), 30, 0, 360, 190, -1)
        else:
            n_specks = 15 if pathology == "MALIGNANT" else 6
            for _ in range(n_specks):
                cv2.circle(img, (cx + np.random.randint(-12, 12), cy + np.random.randint(-12, 12)), 1, 250, -1)

        # Film noise & scanner artifacts
        noise = np.random.normal(0, np.random.uniform(3, 14), (h, w)).astype(np.float32)
        noisy = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

        if i % 5 == 0:
            noisy = cv2.GaussianBlur(noisy, (5, 5), 0)
        if i % 7 == 0:
            cv2.line(noisy, (np.random.randint(0, w), 0), (np.random.randint(0, w), h), 235, 1)

        cv2.imwrite(os.path.join(sub, f"CBIS_DDSM_{view}_{pathology}_{abnormality}_{i:03d}.png"), noisy)
    return target_dir

if DATASET_ROOT is None:
    DATASET_ROOT = generate_synthetic_mammograms(os.path.join(OUTPUT_DIR, "synthetic_cbis"))


## 1. Mammography Characterization & Metadata Parsing
We parse file paths, view orientations (`CC` vs `MLO`), lesion classifications (`Mass` vs `Calcification`), and ground truth pathology (`BENIGN` vs `MALIGNANT`).


In [ ]:
# Cell 4: Metadata Aggregation & Class Breakdown
all_img_paths = []
for ext in ("*.png", "*.jpg", "*.jpeg", "*.dcm"):
    all_img_paths.extend(glob.glob(os.path.join(DATASET_ROOT, "**", ext), recursive=True))

meta_list = []
for p in all_img_paths:
    fname = Path(p).name
    parent = Path(p).parent.name
    
    view = "MLO" if "mlo" in fname.lower() or "mlo" in parent.lower() else "CC"
    pathology = "MALIGNANT" if "malignant" in fname.lower() or "malignant" in parent.lower() else "BENIGN"
    abnormality = "Calcification" if "calc" in fname.lower() or "calc" in parent.lower() else "Mass"

    meta_list.append({
        "image_id": fname,
        "file_path": p,
        "ground_truth_label": pathology,
        "abnormality_type": abnormality,
        "mammographic_view": view
    })

df_meta = pd.DataFrame(meta_list)
print(f"Total Cohort Mammograms: {len(df_meta)}")
print("\nPathology Breakdown:")
print(df_meta["ground_truth_label"].value_counts())
print("\nAbnormality Breakdown:")
print(df_meta["abnormality_type"].value_counts())
print("\nView Breakdown:")
print(df_meta["mammographic_view"].value_counts())


## 2. Image Quality Assessment (IQA) for Mammography
Mammography quality is heavily governed by:
- **Sharpness:** Identifying fine microcalcification clusters without blurring.
- **Dynamic Range / Contrast:** Discerning hypodense/hyperdense lesions through dense glandular background.
- **Noise:** Measuring digitized film grain via high-pass MAD.


In [ ]:
# Cell 5: Mammography IQA Functions
def compute_laplacian_variance(gray):
    return float(np.var(cv2.Laplacian(gray, cv2.CV_64F)))

def compute_tenengrad(gray):
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.mean(gx**2 + gy**2))

def compute_rms_contrast(gray):
    norm = gray.astype(np.float64) / 255.0
    return float(np.std(norm))

def compute_michelson_contrast(gray):
    p1 = float(np.percentile(gray, 1))
    p99 = float(np.percentile(gray, 99))
    return float((p99 - p1) / (p99 + p1 + 1e-6))

def compute_shannon_entropy(gray):
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    return float(entropy(hist, base=2))

def compute_snr_mad(gray):
    mean_val = float(np.mean(gray))
    lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    mad = float(np.median(np.abs(lap - np.median(lap))))
    sigma_n = mad / 0.6745
    if sigma_n < 1e-6:
        return 45.0
    snr = mean_val / (sigma_n + 1e-6)
    return float(np.clip(20.0 * np.log10(max(snr, 1e-6)), -10.0, 50.0))

def extract_mammography_iqa(path):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return None
    
    lap = compute_laplacian_variance(gray)
    ten = compute_tenengrad(gray)
    rms = compute_rms_contrast(gray)
    mic = compute_michelson_contrast(gray)
    ent = compute_shannon_entropy(gray)
    snr = compute_snr_mad(gray)
    
    # Mammographic density ratio (tissue > 120 / all tissue)
    tissue_mask = gray > 15
    dense_ratio = float(np.sum((gray > 120) & tissue_mask) / (np.sum(tissue_mask) + 1e-6))

    norm_lap = np.clip(np.log1p(lap) / 8.0, 0.0, 1.0)
    norm_contrast = np.clip(rms * 4.0, 0.0, 1.0)
    norm_snr = np.clip((snr - 5.0) / 25.0, 0.0, 1.0)
    norm_ent = np.clip(ent / 8.0, 0.0, 1.0)
    composite = float(0.30 * norm_lap + 0.25 * norm_contrast + 0.25 * norm_snr + 0.20 * norm_ent)
    
    tier = "High_Quality" if composite >= 0.65 else ("Medium_Quality" if composite >= 0.45 else "Degraded")
    
    return {
        "laplacian_variance": lap,
        "tenengrad_gradient": ten,
        "rms_contrast": rms,
        "michelson_contrast": mic,
        "shannon_entropy": ent,
        "snr_db": snr,
        "dense_tissue_ratio": dense_ratio,
        "composite_quality_score": composite,
        "quality_tier": tier
    }


In [ ]:
# Cell 6: Run Feature Extraction on Mammograms
sample_n = min(len(df_meta), 300)
eval_df = df_meta.head(sample_n).copy()

iqa_rows = []
for idx, row in eval_df.iterrows():
    m = extract_mammography_iqa(row["file_path"])
    if m:
        m["image_id"] = row["image_id"]
        iqa_rows.append(m)

df_iqa = pd.DataFrame(iqa_rows)
df_full = pd.merge(eval_df, df_iqa, on="image_id")

# Simulate CNN model confidence
np.random.seed(42)
df_full["cnn_confidence"] = np.clip(0.58 + 0.32 * df_full["composite_quality_score"] + np.random.normal(0, 0.04, len(df_full)), 0.42, 0.99)

print("Mammography IQA Feature Extraction Completed.")
df_full[["ground_truth_label", "abnormality_type", "mammographic_view", "laplacian_variance", "rms_contrast", "composite_quality_score"]].head()


## 3. Statistical Analysis & Visual Inspection
We analyze metric spreads across views (`CC` vs `MLO`), compare mass vs calcification contrast, and inspect high-fidelity vs film-noise degraded mammograms.


In [ ]:
# Cell 7: Statistical Summary & Radiographic Visualizations
print("--- CBIS-DDSM Mammography Descriptive Statistics ---")
display(df_full[["laplacian_variance", "tenengrad_gradient", "rms_contrast", "shannon_entropy", "snr_db", "composite_quality_score"]].describe())

# 1. Pathology & Abnormality Distribution
plt.figure(figsize=(8, 4))
sns.countplot(data=df_full, x="ground_truth_label", hue="abnormality_type", palette="Set1")
plt.title("CBIS-DDSM: Pathology Distribution by Abnormality Type", fontsize=12, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "01_cbis_class_distribution.png"), dpi=200)
plt.show()

# 2. Metric Distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_full["laplacian_variance"], kde=True, ax=axes[0, 0], color="navy")
axes[0, 0].set_title("Sharpness (Laplacian Variance)", fontweight="bold")

sns.histplot(df_full["rms_contrast"], kde=True, ax=axes[0, 1], color="darkcyan")
axes[0, 1].set_title("RMS Contrast", fontweight="bold")

sns.histplot(df_full["shannon_entropy"], kde=True, ax=axes[1, 0], color="purple")
axes[1, 0].set_title("Shannon Entropy (bits)", fontweight="bold")

sns.histplot(df_full["snr_db"], kde=True, ax=axes[1, 1], color="coral")
axes[1, 1].set_title("Signal-to-Noise Ratio (dB)", fontweight="bold")
plt.suptitle("CBIS-DDSM: Radiographic Quality Metric Distributions", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "02_cbis_metric_distributions.png"), dpi=200)
plt.show()

# 3. Contrast by Abnormality & View
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_full, x="abnormality_type", y="rms_contrast", hue="mammographic_view", palette="Pastel2")
plt.title("Contrast Dynamic Range by Abnormality & View", fontsize=12, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "03_contrast_by_view.png"), dpi=200)
plt.show()

# 4. Best vs Degraded Mammograms Montage
sorted_df = df_full.sort_values(by="composite_quality_score", ascending=False)
best_rows = sorted_df.head(3)
worst_rows = sorted_df.tail(3)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, (_, row) in enumerate(best_rows.iterrows()):
    im = cv2.imread(row["file_path"], cv2.IMREAD_GRAYSCALE)
    axes[0, i].imshow(im, cmap="gray")
    axes[0, i].set_title(f"HIGH FIDELITY\nScore: {row['composite_quality_score']:.2f} | Lap: {row['laplacian_variance']:.1f}", color="green", fontweight="bold")
    axes[0, i].axis("off")

for i, (_, row) in enumerate(worst_rows.iterrows()):
    im = cv2.imread(row["file_path"], cv2.IMREAD_GRAYSCALE)
    axes[1, i].imshow(im, cmap="gray")
    axes[1, i].set_title(f"NOISY / LOW CONTRAST\nScore: {row['composite_quality_score']:.2f} | Lap: {row['laplacian_variance']:.1f}", color="red", fontweight="bold")
    axes[1, i].axis("off")

plt.suptitle("Visual Comparison: High-Fidelity vs Film-Degraded Mammograms", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "04_best_vs_worst_mammograms.png"), dpi=200)
plt.show()


## 4. Learned Decision Tree: Quality-Aware Adaptive Clinical Routing
We train an explainable Decision Tree classifier that fuses the **IQA feature vector** with **CNN prediction confidence** to make automated, safety-critical clinical routing decisions.


In [ ]:
# Cell 8: Train Learned Decision Tree & Export Routing Metadata
def assign_clinical_routing_ground_truth(row):
    q = row["composite_quality_score"]
    conf = row["cnn_confidence"]
    lap = row["laplacian_variance"]
    contrast = row["rms_contrast"]
    snr = row["snr_db"]
    
    if q < 0.25 or lap < 20.0:
        return "Reject_Reacquire"
    elif (contrast < 0.12 or snr < 11.0) and lap >= 30.0:
        return "Route_Image_Enhancement"
    elif q >= 0.60 and conf >= 0.80:
        return "Accept_Direct_Prediction"
    else:
        return "Expert_Human_Review"

df_full["clinical_decision_target"] = df_full.apply(assign_clinical_routing_ground_truth, axis=1)

feature_cols = ["composite_quality_score", "laplacian_variance", "rms_contrast", "shannon_entropy", "snr_db", "cnn_confidence"]
X = df_full[feature_cols].values
y = df_full["clinical_decision_target"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y if len(np.unique(y)) > 1 else None)

dt_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=3, criterion="gini", random_state=42)
dt_model.fit(X_train, y_train)

acc = dt_model.score(X_test, y_test)
print(f"Learned Decision Tree Test Accuracy: {acc * 100:.2f}%
")

print("--- Decision Logic Rules ---")
print(export_text(dt_model, feature_names=feature_cols))

imp_df = pd.DataFrame({"Feature": feature_cols, "Importance": dt_model.feature_importances_}).sort_values(by="Importance", ascending=False)
print("--- Feature Importances ---")
display(imp_df)

output_csv = os.path.join(OUTPUT_DIR, "cbis_ddsm_quality_and_routing_metadata.csv")
df_full.to_csv(output_csv, index=False)
print(f"[SUCCESS] Exported enriched clinical metadata for CNN & Tree training to: {output_csv}")
